<a href="https://colab.research.google.com/github/devarsh-mavani-19/facenet-notebook/blob/main/facenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install facenet-pytorch
!pip install matplotlib

In [2]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

mtcnn = MTCNN(
    image_size=160,
    margin=20,
    device=device
)

model = InceptionResnetV1(
    pretrained='vggface2'
).eval().to(device)

print("Loaded FaceNet model")

Loaded FaceNet model


In [4]:
def get_embedding(image_path):
    img = Image.open(image_path).convert('RGB')

    face = mtcnn(img)

    if face is None:
        print("No face detected in", image_path)
        return None

    face = face.unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(face)

    return embedding.cpu().numpy()

In [ ]:
emb1 = get_embedding("/content/steve3.jpeg")
emb2 = get_embedding("/content/steve2.webp")


similarity = cosine_similarity(emb1, emb2)[0][0]

print("Cosine similarity:", similarity)

if similarity > 0.7:
    print("Same person")
else:
    print("Different people")

In [ ]:
def show_image(path):
    img = Image.open(path)
    plt.imshow(img)
    plt.axis('off')

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
show_image("/content/steve2.webp")

plt.subplot(1,2,2)
show_image("/content/steve.jpeg")

plt.show()

In [5]:
!pip install facenet-pytorch faiss-cpu tqdm

In [ ]:
!git clone https://github.com/deepinsight/insightface.git

In [6]:
!pip install kaggle

In [7]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"devarshmavani","key":"114f1d46f16603bcef4b5692e4e76de1"}'}

In [8]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle datasets download -d hereisburak/pins-face-recognition

Dataset URL: https://www.kaggle.com/datasets/hereisburak/pins-face-recognition
License(s): CC0-1.0
pins-face-recognition.zip: Skipping, found more recently modified local copy (use --force to force download)


In [10]:
!unzip pins-face-recognition.zip

Archive:  pins-face-recognition.zip
replace 105_classes_pins_dataset/pins_Adriana Lima/Adriana Lima0_0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [11]:
import os
import torch
import numpy as np
import faiss

from PIL import Image
from tqdm import tqdm
from facenet_pytorch import MTCNN, InceptionResnetV1

In [18]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

dataset_path = "/content/105_classes_pins_dataset"

embeddings = []
labels = []
image_paths = []

people = sorted(os.listdir(dataset_path))#[:20]   # limit to first 20 people

for person in tqdm(people):

    person_dir = os.path.join(dataset_path, person)

    if not os.path.isdir(person_dir):
        continue

    for img_name in os.listdir(person_dir):

        img_path = os.path.join(person_dir, img_name)

        try:
            img = Image.open(img_path).convert("RGB")

            face = mtcnn(img)

            if face is None:
                continue

            face = face.unsqueeze(0).to(device)

            with torch.no_grad():
                emb = model(face)

            embeddings.append(emb.cpu().numpy()[0])
            labels.append(person)
            image_paths.append(img_path)

        except:
            pass

embeddings = np.array(embeddings).astype("float32")

print("Total embeddings:", embeddings.shape)
print("Total people:", len(set(labels)))

100%|██████████| 105/105 [1:24:01<00:00, 48.02s/it]

Total embeddings: (17485, 512)
Total people: 105


In [19]:
import faiss

dimension = 512

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Indexed faces:", index.ntotal)

Indexed faces: 17485


In [20]:
def search_face(image_path, k=5):

    img = Image.open(image_path).convert("RGB")

    face = mtcnn(img)
    face = face.unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(face)

    query = emb.cpu().numpy().astype("float32")

    faiss.normalize_L2(query)

    distances, indices = index.search(query, k)

    results = []

    for i in range(k):
        results.append((image_paths[indices[0][i]], distances[0][i]))

    return results

In [15]:
from google.colab import files
files.upload()

Saving bill.webp to bill (1).webp


{'bill (1).webp': b'RIFF\xd0\x13\x00\x00WEBPVP8 \xc4\x13\x00\x00\xb0\\\x00\x9d\x01*\xc1\x00\x05\x01>\x95D\x9dK%\xa3\xbf\xb4%\xf3\xeb\xbb\xf0\x12\x89gn\x91\xcc\xaa\x01\x1b\x1d\x86\x1c\xb7\x9c\xdb\x9f\x0b\xb2\x83\xd4\x08\xab\\\x8fe\x0f\xef|=\x9aP6Yh\xfd(\xcb9\xe1\x7f\xe0\xf3:\xf1,\xb1E.\xe9\x146\x96\x0ec\xbd7\xb83{\xa6ZC8\xf2\x9a\x81\x02\xc7x~\x83\xe8o}\x82\xbd\x8f\xf5\x01\xfe\xa5\xd1\xb6\x0f\x8a\x05m\xf4F\x853\x8b\xe6Y\xe9\x19\xa8\x8a\x18|\xba\xfb\x0cL\xe1\x8f\x88b\x15\x13uvs\xa1\xcb\xed\x0b:~\xe4\x84\xeaXIz\xdb\x0cK\xd36B\xad\x0c\x0f\x02\xca\xcbE\x80\xff\xee`\xea\x97h\xda(\xdd\x85H\xe1\x9e<\x0fI\x87\x92\xc0\xa3\xa4\x90|\x0b\xc3\xecH\xbej\xdb\xaeT\xbd\x17\xa7,\xaf\xd4\xbaM\xd6=\xe4a8\xd1\x9f%\xcf\xc2\xc9\xc7l\xdb\r\xd9g\xc4^\xe6fm\xc3\x83\xb60\xbe\xc6\xbb\x0b3\xa1W9Q\xcd\x84\x10x:\x84Rbgt.a|\xec\xca!\x1e\x15\xfd\x0b\xf1.6%.I=\r\x9b\xaa\xa9\xb7\xfd\xac\xb4:\xfbR\xb1\xcc\x1a\x1e\xfa\xf7\x12M\x0f\xc8\x83\xb6\xf0\xf7\xdf\xd7\xaa\xc5n<\x8e\x08\x8c#L\xae\xa8Z\xa7OypO\xaab^j\x87\xc2\x1dA2\xbb\

In [21]:
results = search_face("/content/bill.webp")

for path, score in results:
    print(score, path)

0.9715619 /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates221_558.jpg
0.95927924 /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates89_614.jpg
0.92400056 /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates45_580.jpg
0.9202054 /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates5_583.jpg
0.89973575 /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates49_582.jpg


In [22]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 11.5 MB/s eta 0:00:00


In [23]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

In [24]:
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name="faces",
    vectors_config=VectorParams(
        size=512,
        distance=Distance.COSINE
    )
)

True

In [25]:
from qdrant_client.models import PointStruct

points = []

for i, emb in enumerate(embeddings):

    points.append(
        PointStruct(
            id=i,
            vector=emb.tolist(),
            payload={
                "person": labels[i],
                "image_path": image_paths[i]
            }
        )
    )

client.upsert(
    collection_name="faces",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
def search_face(image_path, k=5):

    img = Image.open(image_path).convert("RGB")

    face = mtcnn(img)
    face = face.unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(face)

    query = emb.cpu().numpy()[0]

    results = client.query_points(
        collection_name="faces",
        query=query.tolist(),
        limit=k
    )

    for r in results.points:
        print(
            "score:", r.score,
            "person:", r.payload["person"],
            "image:", r.payload["image_path"]
        )

In [31]:
search_face("/content/bill.webp")

score: 0.971561879070217 person: pins_Bill Gates image: /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates221_558.jpg
score: 0.9592791950653761 person: pins_Bill Gates image: /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates89_614.jpg
score: 0.92400065569156 person: pins_Bill Gates image: /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates45_580.jpg
score: 0.9202054767396248 person: pins_Bill Gates image: /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates5_583.jpg
score: 0.8997357472623074 person: pins_Bill Gates image: /content/105_classes_pins_dataset/pins_Bill Gates/Bill Gates49_582.jpg
